# 04 — Pandas Series basics

**What this notebook is about:** so far everything you've written has used plain Python — lists, for-loops, accumulators. That works, but it's slow on real data, verbose to write, and not how the `engine/` code is built. The codebase runs on **pandas**, and the smallest pandas building block is a **Series**.

**One-sentence definition:** a Series is a list with a label attached to each value, plus a pile of built-in operations that work on the whole thing at once instead of one element at a time.

**The big new idea:** *vectorisation*. Instead of writing a for-loop to add 1 to every element, you just say `s + 1` and pandas does the loop for you, in C, much faster. Most of your for-loops from notebooks 02–03 will disappear in pandas code.

**How to use this notebook:** same as the R2 refresher. Predict before running. Type your own code. If a cell errors, read the error before assuming it's broken — pandas errors are usually literal.

**The bridge from notebook 03:** functions and dicts don't go away. Most of what you'll see is *a function that takes a Series in and returns a Series or a dict out*. The accumulator pattern you drilled gets replaced by a single method call — `s.sum()` instead of a `for n in s: total += n` loop.

---
## 1. What a Series is

Compare these two:

```python
prices_list   = [15000, 15050, 15020, 15080, 15100]
prices_series = pd.Series([15000, 15050, 15020, 15080, 15100])
```

Same numbers, two containers. The differences:

1. The Series carries a **label** alongside each value. By default the labels are just `0, 1, 2, 3, 4` — same as a list — but they can be dates, names, anything.
2. The Series knows it's numeric. Operations like `series + 1` or `series * 2` work on the whole thing at once. On a plain list, `[15000, 15050] + 1` is an error.
3. The Series has hundreds of built-in methods: `.sum()`, `.mean()`, `.max()`, `.std()`, etc. No loops needed.

**The label is called the *index*.** This is the single most important word to learn in pandas. The index is what makes a Series more than just a list.

In [ ]:
import pandas as pd   # the convention is to import pandas as `pd`. You'll see this everywhere.

prices = pd.Series([15000, 15050, 15020, 15080, 15100])
print(prices)

Read the output carefully. Two columns:
- **Left column = the index** (here: `0, 1, 2, 3, 4`). These are the labels.
- **Right column = the values** (the prices).

Plus a footer line saying `dtype: int64` — pandas knows these are 64-bit integers.

You can give the index meaningful labels when you build the Series:

In [ ]:
prices_named = pd.Series(
    [15000, 15050, 15020, 15080, 15100],
    index=["Mon", "Tue", "Wed", "Thu", "Fri"],
)
print(prices_named)

Now the left column is days of the week. The values are still the same numbers. This is how the real NQ data works: the index is timestamps (`2017-01-03 09:30`, ...), the values are prices.

### Exercise 1 — build your first Series

Build a Series called `closes` containing the values `[100, 102, 101, 105, 103]` with index `["Mon", "Tue", "Wed", "Thu", "Fri"]`. Print it. Predict what the two columns will look like before running.

In [ ]:
closes = pd.Series(
    [100, 102, 101, 105, 103],                 # <-- Added a comma here
    index=["Mon", "Tue", "Wed", "Thu", "Fri"]   # <-- Changed 'series' to 'Series'
)

print(closes)


---
## 2. Two ways to access a value: `.loc` (by label) and `.iloc` (by position)

Lists have one way: `my_list[0]` gives you the first element. Series have **two**, because the index isn't always 0, 1, 2, 3...

- `series.loc["Wed"]` — look up by **label**. (`.loc` = locate by label.)
- `series.iloc[2]` — look up by **integer position**. (`.iloc` = integer locate.)

When the index is `0, 1, 2, ...` they happen to give the same answer. When the index is dates or names, they're very different.

In [ ]:
print("by label  (Wed):", prices_named.loc["Wed"])
print("by position (2):", prices_named.iloc[2])

# First and last by position — common pattern:
print("first:", prices_named.iloc[0])
print("last :", prices_named.iloc[-1])

**The trap:** `series[0]` *without* `.loc` or `.iloc` is ambiguous and pandas's behaviour for it has changed across versions. **Always use `.loc` or `.iloc` explicitly.** It costs four extra characters and removes a whole class of bugs.

**Mental rule of thumb:**
- Need a specific named thing (a particular date, a particular ticker)? → `.loc`
- Need "the first one", "the last one", "the third one"? → `.iloc`

### Exercise 2 — access values both ways

Using your `closes` Series from Exercise 1:

1. Print Wednesday's close using `.loc`.
2. Print Wednesday's close using `.iloc`. (What position is Wednesday at?)
3. Print Friday's close using `.iloc[-1]`. (Why does `-1` work?)


In [ ]:
closes = pd.Series(
    [100, 102, 101, 105, 103],
    index=["Mon", "Tue", "Wed", "Thu", "Fri"],
)

print("by label  (Wed):", closes.loc["Wed"])
print("by position (2):", closes.iloc[2])
print("by position (-1):", closes.iloc[-1])


---
## 3. Vectorised arithmetic — the loop disappears

Here's the move that makes pandas worth learning. Compare:

```python
# Old way — plain Python loop
prices = [100, 102, 101, 105, 103]
prices_plus_1 = []
for p in prices:
    prices_plus_1.append(p + 1)

# Pandas way — one line, no loop
prices = pd.Series([100, 102, 101, 105, 103])
prices_plus_1 = prices + 1
```

Both produce the same answer. The pandas version is shorter, faster, and reads more like maths.

**The rule:** any arithmetic operation between a Series and a single number is applied to *every* element. `s + 1`, `s * 2`, `s / 100`, `s ** 2` — all element-wise, no loop needed.

Operations between **two Series** are also element-wise, paired up by *label* (not position — this matters).

In [ ]:
prices = pd.Series([100, 102, 101, 105, 103])

print("prices       :", prices.tolist())
print("prices + 1   :", (prices + 1).tolist())
print("prices * 2   :", (prices * 2).tolist())
print("prices / 100 :", (prices / 100).tolist())

In [ ]:
# Two Series — element-wise, paired by index position here (because both default 0..4)
highs = pd.Series([105, 107, 103, 110, 108])
lows  = pd.Series([ 98,  99,  97, 102, 100])

ranges = highs - lows
print(ranges)

### Exercise 3 — daily returns (generic)

A *return* is the percentage change from yesterday's close to today's close. The formula:

$$\text{return}_t = \frac{\text{close}_t - \text{close}_{t-1}}{\text{close}_{t-1}}$$

Use your `closes` Series. Compute the returns *for all five days at once*, no loop. Hint: pandas has a built-in `.pct_change()` method that does exactly this. Print the result.

Predict: what's special about the first value of the returned Series? (Hint: what's the "previous close" for the very first day?)

In [ ]:
import pandas as pd

closes = pd.Series(
    [100, 102, 101, 105, 103],
    index=["Mon", "Tue", "Wed", "Thu", "Fri"]
)

# Calculate the daily percentage returns
daily_returns = closes.pct_change()

# Print the final result
print(daily_returns)

---
## 4. Built-in summaries — accumulators, gone

Every accumulator pattern you wrote in notebook 02–03 has a one-liner pandas equivalent.

| Accumulator loop | Pandas one-liner |
|---|---|
| sum | `s.sum()` |
| count | `s.count()` or `len(s)` |
| average | `s.mean()` |
| max / min | `s.max()` / `s.min()` |
| std dev | `s.std()` |
| count of values matching a condition | `(s > 0).sum()` ← read this carefully |

The last one deserves a closer look — it's the key to filtering.

In [ ]:
pnls = pd.Series([100, -50, 200, -150, 80, -30, 120])

print("sum      :", pnls.sum())
print("count    :", pnls.count())
print("mean     :", pnls.mean())
print("max      :", pnls.max())
print("min      :", pnls.min())
print("std      :", pnls.std())

**Compare to notebook 03's `summarise_trade`** — most of that function's body is now one line per measurement, with no loops.

---
## 5. Boolean Series — the key to filtering

This is the single weirdest thing in pandas the first time you see it. Stick with it — it unlocks everything.

When you write `pnls > 0`, pandas does *not* give you back a single True/False. It gives you back a whole **Series of True/False values**, one per element:

In [ ]:
pnls = pd.Series([100, -50, 200, -150, 80, -30, 120])

mask = pnls > 0
print(mask)

Read the output. `mask` is a Series the same length as `pnls`, with `True` where `pnls` was positive and `False` where it wasn't. This is called a **boolean mask** or just a **mask**.

**Two things you can do with a mask:**

1. **Filter** — pass it back into the original Series to get only the matching rows: `pnls[mask]` or `pnls[pnls > 0]`.
2. **Count** — call `.sum()` on it. Python treats `True` as 1 and `False` as 0, so `(pnls > 0).sum()` counts the trues.

These two patterns will appear everywhere in the engine.

In [ ]:
# Filter — get just the winning trades
winners = pnls[pnls > 0]
print("winners:")
print(winners)

# Count — how many winners?
print("\nn_winners:", (pnls > 0).sum())

# Sum of just the winners (this is gross_profit from notebook 03 E4)
print("gross_profit:", pnls[pnls > 0].sum())

**Pause and re-read that last cell.** Compare to notebook 03 E4 (`summarise_trade`). What took you a 10-line for-loop with two trackers and an if/elif is now three lines, no loops, no trackers. **This is what "vectorisation" pays off.**

### Exercise 4 — count and sum with masks

Using `pnls = pd.Series([100, -50, 200, -150, 80, -30, 120])`:

1. Count how many trades were strictly less than 0 (losers). Use a mask + `.sum()`.
2. Sum the losing trades only (this is the raw, negative `gross_loss`).
3. Compute `gross_loss` as a positive number (i.e. wrap step 2 in `abs()`).
4. Predict each answer before running.

**No for-loops.** If you find yourself reaching for one, stop and think about which mask + method combo would do it.

In [ ]:
import pandas as pd 

pnls = pd.Series([100, -50, 200, -150, 80, -30, 120])
mask = pnls < 0

loosers = pnls[pnls < 0]
print("\nn_loosers:" , (pnls < 0).sum())
print("gross_loss:" , pnls[pnls < 0].sum())

abs(pnls[mask].sum())


In [ ]:
closes = pd.Series(
    [100, 102, 101, 105, 103],
    index=["Mon", "Tue", "Wed", "Thu", "Fri"],
)

prev_close = closes.shift(1)
print("closes:")
print(closes)
print("\nprev_close (shifted by 1):")
print(prev_close)

Read the two outputs side by side:

- On `Mon`, `closes` is 100 and `prev_close` is `NaN` — short for "not a number," pandas's marker for missing data. There is no previous day for the first row, so the answer is honestly missing.
- On `Tue`, `closes` is 102 and `prev_close` is 100 — Monday's close, now sitting on Tuesday's row.
- ... and so on.

Now `closes - prev_close` gives you the daily change, with each row using only information that would have been known at that row's time. That's lookahead-safe.

**NaN propagation:** any arithmetic with `NaN` produces `NaN`. So `(closes - prev_close)` on Mon is `NaN`. This is correct — you genuinely don't have a daily-change measurement for the first day. Most pandas methods (`.mean()`, `.sum()`) skip NaNs by default.

### Exercise 5 — daily change using `.shift(1)`

Using `closes = pd.Series([100, 102, 101, 105, 103], index=["Mon", "Tue", "Wed", "Thu", "Fri"])`:

1. Compute `daily_change = closes - closes.shift(1)`. Print it. Predict each value before running.
2. Compute the average daily change with `.mean()`. Does pandas include or exclude the NaN? (Run it and see.)
3. Bonus: compute the daily change as a percentage instead. Two ways to do it — one with `.shift(1)` and one with `.pct_change()`. Confirm they give the same answer.

In [ ]:
closes = pd.Series(
    [100, 102, 101, 105, 103  ],
    index =["Mon", "Tue", "Wed", "Thu", "Fri"]
)

daily_change = closes - closes.shift(1)
print(daily_change)

avg_change = daily_change.mean()
print(avg_change)

---
## 7. Functions that take a Series and return a dict

Putting it together. The R2 / notebook 03 pattern — "function that takes a sequence and returns a dict of summaries" — works on a Series too. The body just shrinks dramatically because the loops are gone.

In [ ]:
# Compare this to notebook 03 E4. Same output, far less code.
def summarise_trades(pnls):
    gross_profit = pnls[pnls > 0].sum()
    gross_loss = abs(pnls[pnls < 0].sum())
    return {
        "n_trades": len(pnls),
        "n_winners": (pnls > 0).sum(),
        "total_pnl": pnls.sum(),
        "gross_profit": gross_profit,
        "gross_loss": gross_loss,
        "profit_factor": gross_profit / gross_loss if gross_loss > 0 else 0.0,
    }

test_pnls = pd.Series([100, -50, 200, -150, 80, -30, 120])
print(summarise_trades(test_pnls))

Output should match your notebook 03 E4 exactly. The two functions are doing the same thing — this version is just written in pandas instead of plain Python.

### Exercise 6 — `summarise_returns(returns)` (quant)

Write a function `summarise_returns(returns)` that takes a Series of daily returns (positive or negative decimals like 0.012 or -0.008) and returns a dict with:

- `n_days` — how many days
- `mean_return` — average return
- `volatility` — standard deviation of returns (use `.std()`)
- `n_up_days` — count of days with strictly positive return
- `n_down_days` — count of days with strictly negative return
- `best_day` — max return
- `worst_day` — min return

**Constraint:** no for-loops. Every value should be a one-line pandas expression.

Test it on:

```python
test_returns = pd.Series([0.012, -0.008, 0.020, -0.015, 0.008, -0.003, 0.011])
```

Predict each key's value before running.

In [ ]:
def summarise_returns(returns):
    return {
        "n_days": len(returns),
        "mean_return": returns.mean(),
        "volatility": returns.std(),
        "n_up_days": (returns > 0).sum(),
        "n_down_days": (returns < 0).sum(),
        "best_day": returns.max(),
        "worst_day": returns.min(),
    }

test_returns = pd.Series([0.012, -0.008, 0.020, -0.015, 0.008, -0.003, 0.011])

print(summarise_returns(test_returns))


---
## 8. The traps — what bites people first

A short list of pandas mistakes you will almost certainly make at least once. Recognising them up front saves hours.

1. **`series[0]` vs `series.loc[0]` vs `series.iloc[0]`.** If your index is `0, 1, 2, ...` they happen to agree. If it's dates or strings, `series[0]` is ambiguous and may error or silently do the wrong thing. **Always use `.loc` or `.iloc` explicitly.**

2. **Forgetting `.shift(1)` on a derived feature.** The number-one source of lookahead bias. If your feature involves anything from a *previous* bar, the previous-bar value must come from `.shift(1)`, not from raw indexing. This is a CLAUDE.md hard rule.

3. **`NaN` quietly contaminating everything.** Any arithmetic with `NaN` gives `NaN`. Most reductions (`.mean()`, `.sum()`) skip NaNs by default, but some (`.cumsum()`) do not. If a result looks suspiciously empty or wrong, check for NaNs first with `series.isna().sum()`.

4. **`and` / `or` between Series.** Plain Python's `and` and `or` don't work on Series — pandas raises `ValueError: The truth value of a Series is ambiguous.` You need element-wise operators: `&` and `|`, with each side in parentheses: `(s > 0) & (s < 100)`. This will trip you up; just remember it.

5. **Modifying a Series you got back from a filter or slice.** Sometimes pandas warns about a `SettingWithCopyWarning`. The safe rule is: if you want a modified version, create it explicitly with `.copy()` first. We'll cover this properly when it matters.

---
## Done — what's next

If Exercise 6 worked, you can now do everything notebook 03's boss fight did, but in pandas idiom:

- Building a Series with an index ✅
- `.loc` vs `.iloc` ✅
- Vectorised arithmetic (no loop) ✅
- Boolean masks for filtering and counting ✅
- `.shift(1)` for lookahead safety ✅
- Function takes Series, returns dict ✅

**Tell me how it went:**

1. Does the no-loop style feel natural yet, or does your hand still reach for `for n in pnls:` first?
2. Does `.loc` vs `.iloc` feel solid, or are you guessing which to use?
3. Did `.shift(1)` make sense — why does the first value have to be NaN?
4. Did the mask idea (`pnls[pnls > 0]`) click, or is it still strange?

**Next up: notebook 05 — pandas DataFrame basics.** A DataFrame is a stack of Series sharing the same index — that's literally what `data/nq_15m_data.csv` becomes when you load it. Same operations you just learned, now applied across columns at once.